# Part 3 — Add-on to the agentic system

A few capabilities are already in this repo but weren't covered in Part 2:

1. **The stdio transport** — the client spawns the server; no terminal needed. This is an alternative to the http-streamable server. 
2. **Swapping the LLM backend** (Using open-weight models)
3. **Skills** — recipes the agents load on demand
4. **Follow-up queries** — continue a run instead of starting cold
5. **Memory** — lessons that persist across runs - helping agent get better with more interactions.

Unlike Part 2, this notebook needs **no server running in a terminal**

## 1. stdio server as an alternative way to host servers

Part 2 connected to a URL (`streamable-http`): the server was a process *you*
started and could watch. Production MCP clients (e.g. Claude Desktop) can also *spawn a server as a subprocess* and talks to it over stdin/stdout. Same server code, zero changes; only the client-side config differs.

Everything below runs over stdio. If you still have the Part 2 server running in a terminal, it is simply not used here.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

from dotenv import load_dotenv

load_dotenv(Path.cwd().parent / ".env")

OUTPUT_DIR = str((Path.cwd().parent / "agent-output").resolve())
SERVER_REPO = str((Path.cwd().parent.parent / "spectra-mcp-server").resolve())

from agents import build_graph, follow_up, load_tools, make_llm, new_run

STDIO_CONFIG = {
    "spectra": {
        "transport": "stdio",
        "command": sys.executable,
        "args": ["-m", "mcp_server", "--transport", "stdio"],
        "cwd": SERVER_REPO,
    }
}

tools = await load_tools(STDIO_CONFIG)
print("tools over stdio:", [t.name for t in tools])

Identical tool list to Part 2. (Each tool call spawns a fresh short-lived server process; that's fine here because our tools are stateless — everything lives in files.)

## 2. Another backend in one argument

Gemini's free tier is ~20 requests/day — about one agent run. Because the
client only speaks the OpenAI chat protocol, we can point it at any
OpenAI-compatible endpoint. [Groq](https://console.groq.com/keys) offers free
keys (email signup, no credit card) for open-weight US models like OpenAI's
`gpt-oss-120b`, at ~14,400 requests/day. `agents/llm.py` ships both configs;
adding a university or lab endpoint is three more lines there. 

In [ ]:
llm = make_llm("groq")   # needs GROQ_API_KEY in .env
llm.invoke("What model or LLM are you").content

## 3. Skills: skills are like recipes (whereas tools are for individual actions)

A **tool** is a capability (`compute_power_spectrum`). A **skill** is the
know-how for using tools well — which order, which conventions, what sanity
checks. Each skill is a markdown file in `skills/` with a two-line header:

```
---
name: cosmology-comparison
description: Recipe for comparing matter power spectra ...
---
...full instructions...
```

The trick is **progressive disclosure**: the lead only ever sees the cheap
name+description index at planning time; the worker pulls the full text with
the `load_skill` tool when a step calls for it. Context stays small, and
adding know-how never requires touching code — drop in a markdown file.

In [ ]:
from agents.skills import load_skill, skill_index

print(skill_index())
print()
print(load_skill.invoke({"name": "cosmology-comparison"})[:600], "...")

## 4. A run that uses the skill

Same graph as Part 2 — the skill index is already in the lead's planning
prompt, and `load_skill` rides along with the MCP tools. 

Note in the plan: the lead usually schedules loading the recipe as its own step.

In [ ]:
graph = build_graph(llm, tools)

TASK = f"""Compare the linear matter power spectrum at z=0 of standard LCDM
against LCDM with total neutrino mass 0.20 eV, including a plot against the
eBOSS DR14 Lyman-alpha data with LCDM as the ratio reference.
Save all files to {OUTPUT_DIR}."""

state = await graph.ainvoke(new_run(TASK))

for step in state["plan"]:
    print(f"plan {step['id']}: {step['description']}")
print()
for i, r in enumerate(state["step_results"], 1):
    print(f"step {i} -> {r}\n")

## 5. Follow-up queries

Part 2 always started cold (each query + response is fresh). `follow_up(previous_state, new_task)` seeds the
next run with a summary of the previous one (task + report, including file
paths), so the agents build on what exists instead of recomputing it. Note the changes in the plan:

In [ ]:
state2 = await graph.ainvoke(follow_up(
    state,
    "Add wCDM with w0=-0.9 to the same comparison figure, reusing the "
    "spectra you already computed where possible."))

for step in state2["plan"]:
    print(f"plan {step['id']}: {step['description']}")
print()
from IPython.display import Markdown
Markdown(state2["final_report"])

In [ ]:
from IPython.display import Image

Image(f"{OUTPUT_DIR}/power_spectrum_comparison.png", width=700)

## 6. Memory across runs

`MEMORY.md` at the repo root is read into the lead's planning prompt at the
start of **every** run; agents append to it with the `remember` tool when they learn something worth keeping. 

So memory of this agentic system is a simple mechanism of saving a file. You can edit it by hand, and you it is recommended to prune it occasionally.

In [ ]:
from agents.memory import read_memory, remember

remember.invoke({"lesson": "plot_power_spectra uses the first file in "
                           "spectrum_files as the ratio reference (reference_index 0)."})
print(read_memory())

The next run's lead will see that line while planning — try re-running the
Part 2 notebook and check the plan wording.

## 7. What else can be these systems robust and production-ready?


1. **A verification process** — after the plot step, check the work: does the PNG exist, are all requested models present, is the physics sane? Agents that check their own output are dramatically more reliable.
2. **Parallel workers** — LangGraph's `Send` API lets the lead dispatch
   independent steps (e.g. the per-model compute calls) concurrently.
3. **Human-in-the-loop** — LangGraph's `interrupt()` + a checkpointer pauses the graph for plan approval and resumes later.
4. **MCP prompts & skills** — MCP's other primitives: the *server* can ship usage recipes (server-side skills) and read-only data, so the know-how travels with the tools to any client.
5. **Observability** — LangSmith/Langfuse tracing of every LLM and tool call; useful for debugging.

### Customization

- Add a function to the server's `tools/` package and watch it appear in the
  client's tool list with zero client changes.
- Write a `skills/your-workflow.md` for a procedure you care about.
- Point `agents/llm.py` at your institution's OpenAI-compatible endpoint.
- Replace the spectra server with an MCP server for *your* problem.